# Vision Sentinel 2.0: full GPU training run

Reproduces the `gpu-full` plan (`training/configs/hyp_gpu_full.yaml`) on a Colab GPU with the same data pipeline, frozen splits and evaluation as the repository. Runtime → Change runtime type → GPU first.

Every number this notebook prints is measured on the frozen test split (`data/manifests/test.txt`, checksum in `test_manifest.sha256`).

In [ ]:
!nvidia-smi
!git clone https://github.com/paramsn37-sketch/Vision_Sentinel_2.0_GMP.git vs2
%cd vs2/backend
!pip -q install uv && uv sync --group train --group eval

## 1. Data
Downloads Construction-PPE and the capped Open Images V7 subset, merges them into the taxonomy, and splits them with the frozen manifests (so the test split is identical to the CPU run).

In [ ]:
!uv run python ../training/scripts/download_construction_ppe.py
!uv run python ../training/scripts/download_openimages.py
!uv run python ../training/scripts/merge.py
!uv run python ../training/scripts/split.py
!sha256sum -c ../data/manifests/test_manifest.sha256 || echo 'test split differs from the frozen one: results are not comparable'

## 2. Baselines (v1 setup and stock nano model, person only)

In [ ]:
!uv run python ../training/scripts/evaluate.py --weights yolo11m.pt --name baseline-v1-yolo11m --coco
!uv run python ../training/scripts/evaluate.py --weights yolo26n.pt --name baseline-yolo26n-coco --coco

## 3. Train
150 epochs, patience 25, cosine LR, AMP. About 2–3 h on a T4 for YOLO26s at 640 px. Swap `model:` in the YAML for `yolo26n.pt` to train the nano model.

In [ ]:
!uv run python ../training/scripts/train.py --hyp ../training/configs/hyp_gpu_full.yaml --name vs2-y26s-gpu640

## 4. Evaluate, pick thresholds, export, benchmark, error analysis

In [ ]:
RUN = "vs2-y26s-gpu640"
!uv run python ../training/scripts/evaluate.py --weights ../training/runs/{RUN}/weights/best.pt --name {RUN} --imgsz 640 --thresholds
!uv run python ../training/scripts/export.py --run {RUN} --name vs2-y26s --imgsz 640
!uv run python ../training/scripts/benchmark.py --weights ../models/vs2-y26s.pt --name vs2-y26s
!uv run python ../training/scripts/error_analysis.py --weights ../models/vs2-y26s.pt
!uv run pytest -m model -q

## 5. Keep the records
Download `models/`, `docs/eval/detector_results.json`, `training/registry.json`, `training/benchmarks/latency.json` and `training/reports/` and commit them (weights under 50 MB only), or copy them to Drive.

In [ ]:
!zip -qr /content/vs2_results.zip ../models ../docs/eval/detector_results.json ../training/registry.json ../training/benchmarks ../training/reports
from google.colab import files

files.download("/content/vs2_results.zip")